# v06 · Character manifold: per-symbol position sub-addresses
`hyperwebster_manifold.py`.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
mf = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster_manifold.py", "mf")
M = mf.CharacterManifold()
rows = []
for L in (140, 1000, 4000):
    t = CORPUS[:L]; ma = M.decompose(t); assert M.reconstruct(ma) == t
    c = Counter(t)
    Mn = math.factorial(L)
    for k in c.values(): Mn //= math.factorial(k)
    floor = Mn.bit_length() - 1
    rows.append((L, round(L*math.log2(97)), ma.total_sub_bits(), floor, ma.n_active))
print("    L   naive_bits  sum(sub_bits)  log2(multinomial)  active_dims")
for r in rows: print("%5d  %10d  %13d  %17d  %11d" % r)

    L   naive_bits  sum(sub_bits)  log2(multinomial)  active_dims
  140         924            775                564           30
 1000        6600           6086               4662           54
 4000       26400          25819              20146           65


`sum(sub_bits)` is the sum of `⌈log₂ C(L, cₖ)⌉` over symbols k: each symbol's positions are ranked among *all* L positions, ignoring the positions already taken by the other symbols. The exact number of distinct strings with these symbol counts is the multinomial, and its log₂ (last-but-one column) is smaller. Notebook 03 builds the exact address.